In [1]:
from datetime import datetime
import pandas as pd
import sys
from pathlib import Path

# Adjust this if your structure differs
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)


_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_reb_predictions.csv")


Project root added: C:\Users\micha\Coding\python\nba_steam\nba_mike


In [2]:
out.to_csv("./test_reb.csv", index=False)
out

,player,team,opp,is_home,pred_reb,baseline_reb,delta_reb,p_over_baseline_2
0,Jarrett Allen,CLE,CHA,0,9.449473,0.0,9.449473,0.995173
1,James Harden,CLE,CHA,0,4.726206,0.0,4.726206,0.922868
2,Jaylon Tyson,CLE,CHA,0,4.276206,0.0,4.276206,0.897709
3,Craig Porter Jr.,CLE,CHA,0,3.098982,0.0,3.098982,0.784910
4,Donovan Mitchell,CLE,CHA,0,2.986206,0.0,2.986206,0.769096
...,...,...,...,...,...,...,...,...
111,Cameron Johnson,DEN,POR,0,3.601206,0.0,3.601206,0.843360
112,Jerami Grant,POR,DEN,1,3.411973,0.0,3.411973,0.823457
113,Bruce Brown,DEN,POR,0,3.338982,0.0,3.338982,0.815126
114,Scoot Henderson,POR,DEN,1,3.218706,0.0,3.218706,0.800544


In [3]:
# ============================
# TEST: Rebounds selectors using test_reb.csv
# ============================

import pandas as pd

from model_training.rebounds.selector import (
    select_jackpot_reb_ticket,
    select_matchup_coverage_reb_ticket,
    assign_pencil_decision_reb,
)

# --- Load uploaded test file ---
df = out.copy()  # Replace with pd.read_csv("test_reb.csv") if loading from file
print(f"Loaded {len(df):,} rows")
print("Columns:", sorted(df.columns.tolist()))

# --- Basic required columns sanity ---
required = {"player","team","opp","is_home","pred_reb","baseline_reb","delta_reb"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required prediction columns: {sorted(missing)}")

# --- Detect baseline prob column ---
p_cols = [c for c in df.columns if c.startswith("p_over_baseline_")]
if not p_cols:
    raise ValueError("No p_over_baseline_* column found.")
p_col = sorted(p_cols)[-1]
print("Using probability column:", p_col)


# ============================================================
# 1) JACKPOT
# ============================================================
jackpot = select_jackpot_reb_ticket(
    df,
    n_legs=3,
    over_baseline_col=p_col,
    min_pred_reb=7.0,
    min_p_over_baseline=0.18,
    min_delta_reb=2.0,
    max_per_team=1,
)

print("\n=== JACKPOT ===")
print(jackpot[["player","team","pred_reb","baseline_reb","delta_reb",p_col]].to_string(index=False))


# ============================================================
# 2) COVERAGE
# ============================================================
coverage = select_matchup_coverage_reb_ticket(
    df,
    players_per_matchup=2,
    insurance_per_matchup=1,
    min_pred_reb=6.0,
    p_col=p_col,
    min_prob=0.18,
    min_delta_reb=0.0,
    max_legs=18,
)

print("\n=== COVERAGE ===")
print(coverage[["player","team","pred_reb","delta_reb",p_col]].to_string(index=False))


# ============================================================
# 3) OVER LINE (if sportsbook column exists)
# ============================================================
if "p_over_9_5" in df.columns:
    over_line = select_over_line_reb_ticket(
        df,
        n_legs=10,
        p_col="p_over_9_5",
        min_pred_reb=6.0,
        min_prob=0.58,
        max_per_team=3,
    )

    print("\n=== OVER LINE ===")
    print(over_line[["player","team","pred_reb","delta_reb","p_over_9_5"]].to_string(index=False))
else:
    print("\n[INFO] No sportsbook probability column (p_over_9_5) found — skipping over-line test.")


# ============================================================
# 4) PENCIL LABELS
# ============================================================


Loaded 116 rows
Columns: ['baseline_reb', 'delta_reb', 'is_home', 'opp', 'p_over_baseline_2', 'player', 'pred_reb', 'team']
Using probability column: p_over_baseline_2

=== JACKPOT ===
         player team  pred_reb  baseline_reb  delta_reb  p_over_baseline_2
Donovan Clingan  POR 12.181482           0.0  12.181482           0.998836
   Nikola Jokić  DEN 11.506206           0.0  11.506206           0.998365
  Jalen Johnson  ATL 10.441206           0.0  10.441206           0.997162

=== COVERAGE ===
           player team  pred_reb  delta_reb  p_over_baseline_2
    Chet Holmgren  OKC  9.091206   9.091206           0.994126
    Jarrett Allen  CLE  9.449473   9.449473           0.995173
      Rudy Gobert  MIN  9.556482   9.556482           0.995445
     Cooper Flagg  DAL  6.893706   6.893706           0.979402
   Daniel Gafford  DAL  6.841206   6.841206           0.978753
  Donovan Clingan  POR 12.181482  12.181482           0.998836
     Nikola Jokić  DEN 11.506206  11.506206           0.

In [4]:
df_labeled = assign_pencil_decision_reb(df)

display("\n=== PENCIL COUNTS ===")
display(df_labeled["pencil"].value_counts())

display("\n=== TOP 15 (by pred_reb) ===")
display(
    df_labeled.sort_values("pred_reb", ascending=False)
    [["player","team","pred_reb","delta_reb",p_col,"pencil"]]
    .head(15)
    
)


'\n=== PENCIL COUNTS ==='

pencil
coverage_only    104
smash              8
over               4
Name: count, dtype: int64

'\n=== TOP 15 (by pred_reb) ==='

,player,team,pred_reb,delta_reb,p_over_baseline_2,pencil
102,Donovan Clingan,POR,12.181482,12.181482,0.998836,smash
103,Nikola Jokić,DEN,11.506206,11.506206,0.998365,smash
41,Jalen Johnson,ATL,10.441206,10.441206,0.997162,smash
23,Jusuf Nurkić,UTA,9.938706,9.938706,0.996293,smash
57,Rudy Gobert,MIN,9.556482,9.556482,0.995445,smash
0,Jarrett Allen,CLE,9.449473,9.449473,0.995173,smash
42,Bam Adebayo,MIA,9.121206,9.121206,0.994222,smash
81,Chet Holmgren,OKC,9.091206,9.091206,0.994126,smash
7,Alex Sarr,WAS,8.626206,8.626206,0.992396,over
24,Kyle Filipowski,UTA,8.251206,8.251206,0.990610,over
